# One model, two exchange formats

**Lesson 2 of 3 | Beginner | About 30 minutes**

## Learning scenario

The engineering model is ready to leave our Python session. A web
application wants AML JSON; an established AutomationML tool wants
an `.aml` XML file. We should not build two models. We should expose
the same validated model through two exchange formats.

This notebook starts from a provided valid motor example so it can
run independently from Lesson 1.


## 1. Load the starting model

`load_json` parses canonical AML JSON into the same `CAEXFile` model
that the Python builders create.


In [ ]:
from automationml import load_json
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "examples").exists():
    repo_root = repo_root.parent

source_path = repo_root / "examples/validation-suite/json/small-valid-motor.json"
document = load_json(source_path)


In [ ]:
motor = document.instance_hierarchies[0].internal_elements[0]

print("Document:", document.file_name)
print("Element:", motor.name)
print("Class:", motor.ref_base_system_unit_path)


## 2. Validate before exchange

Serialization can faithfully preserve an invalid reference. It
cannot decide whether that reference is correct. Validation is a
separate step and should happen before trusted export.


In [ ]:
issues = document.caex_validation_issues(strict_xsd=True)

assert issues == []
print("Ready for exchange: no validation issues.")


## 3. Create canonical AML JSON

`to_aml_json` uses AutomationML/CAEX names such as
`InstanceHierarchy` and `RefBaseSystemUnitPath`. Python's
`snake_case` field names remain an implementation convenience.


In [ ]:
json_text = document.to_aml_json(indent=2)

print("\n".join(json_text.splitlines()[:24]))
print("... preview shortened")


In [ ]:
assert '"InstanceHierarchy"' in json_text
assert '"RefBaseSystemUnitPath"' in json_text
assert '"instance_hierarchies"' not in json_text

print("Canonical CAEX property names confirmed.")


## 4. Parse the JSON back into a model

Parsing is the first half of a round trip. The result is a new
`CAEXFile` object with the same model content.


In [ ]:
from automationml import CAEXFile

from_json = CAEXFile.from_aml_json(json_text)

assert from_json.to_aml_dict() == document.to_aml_dict()
print("AML JSON round trip passed.")


## 5. Create AML XML from the same object

No JSON-to-XML mapping is written in this notebook. The XML adapter
reads the same Pydantic model that produced the JSON string.


In [ ]:
xml_text = document.to_aml_xml(pretty=True)

print("\n".join(xml_text.splitlines()[:18]))
print("... preview shortened")


## 6. Parse the XML and compare model content

Whitespace and text ordering conventions differ between formats.
Canonical AML dictionaries give us a format-neutral comparison.


In [ ]:
from_xml = CAEXFile.from_aml_xml(xml_text)
canonical = document.to_aml_dict()

assert from_json.to_aml_dict() == canonical
assert from_xml.to_aml_dict() == canonical
print("JSON and XML reconstruct the same AutomationML model.")


## 7. Write both exchange files

We use a temporary folder in the lesson so repeated runs do not
leave generated files in the repository. Real applications can
pass any target `Path` to the same functions.


In [ ]:
from tempfile import TemporaryDirectory
from automationml import dump_json, dump_xml

temporary_directory = TemporaryDirectory()
output_directory = Path(temporary_directory.name)

json_path = output_directory / "motor.json"
xml_path = output_directory / "motor.aml"


In [ ]:
dump_json(document, json_path)
dump_xml(document, xml_path)

print(json_path.name, json_path.stat().st_size, "bytes")
print(xml_path.name, xml_path.stat().st_size, "bytes")


## 8. Load both files through the public IO API

This is the full file-level round trip an application would use.


In [ ]:
from automationml import load_xml

json_document = load_json(json_path)
xml_document = load_xml(xml_path)

assert json_document.to_aml_dict() == canonical
assert xml_document.to_aml_dict() == canonical
print("Both files loaded with identical canonical content.")


## 9. One edit, two updated representations

Model changes happen before serialization. We rename a copy of the
instance and then observe the new name in both outputs.


In [ ]:
edited = document.model_copy(deep=True)
edited_motor = edited.instance_hierarchies[0].internal_elements[0]
edited_motor.name = "Mixer Motor"

assert '"Name": "Mixer Motor"' in edited.to_aml_json()
assert 'Name="Mixer Motor"' in edited.to_aml_xml()
print("One model edit reached both exchange formats.")


## Your turn

Change the motor speed on `edited_motor`. Then generate both strings
again and search for the new value. Finally, break its
`RefBaseSystemUnitPath` and compare successful serialization with
the validator's semantic result.

## Takeaways

- The Pydantic `CAEXFile` is the source of truth.
- AML JSON is the primary readable exchange representation.
- AML XML is produced from the same model for existing toolchains.
- Round trips compare model content, not text formatting.
- Validation and serialization answer different questions.
